# acceleration and jerk
"Fit" or estimate a polynomial of time to explain the residuals away from the clock fourier fit; interpret as an acceleration (and maybe in future jerk, snap, crackle, and pop).

## bugs and to-dos:
- This code should not run best_clocks_in_star(); this should read from the clocks database.
- Only does acceleration, no jerk.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt

In [ ]:
# all this to import the KeplerClocks code
from pathlib import Path
import sys
target_dir = Path.cwd() / "../../KeplerClocks/py"
sys.path.append(str(target_dir.resolve()))
import clocks

In [ ]:
kicid = "KIC007917485" # "KIC001573174" # "KIC009899416"
clock_table = clocks.best_clocks_in_star(kicid, write_to_db=False)
print(clock_table)

In [ ]:
def polynomial_time_delay(row, njack=8, frequency_tweak=0., nterms=3):
    """
    - set `nterms` to 3 to get acceleration; 4 for jerk; 2 for drift only.
    """
    # unpack
    kicid, om, M, value = row['star_id'], row['angular_frequency'], row['fourier_series_degree'], row['theoretical_value']

    # adjust
    om += frequency_tweak * om

    # get data
    ts, ys, errs, deltaf, deltat = clocks.get_kepler_data(kicid)
    ivars = 1. / errs ** 2
    X, ms, pars = clocks.fourier_wls_fit(om, M, ts, ys, ivars)
    resids = ys - X @ pars
    deriv_pars = clocks.take_derivative_wrt_phase(pars, ms)
    derivs = om * X @ deriv_pars # dflux / dt

    # make design matrix for acceleration model
    t0 = np.sum(ivars * ts) / np.sum(ivars)  # ivar-weighted mean time
    foo, denominator = [], 1.
    for term in range(nterms):
        foo += [ (ts - t0) ** term * derivs / denominator, ]
        denominator *= (term + 1)
    A = np.vstack(foo).T
    _, d = A.shape

    # start jackknife
    subsamp = np.arange(len(ys)) * njack // len(ys)
    pars = np.zeros((njack, d)) + np.nan
    for samp in range(njack):
        loo = (subsamp != samp)
        Aloo = A[loo, :]
        ATA = Aloo.T @ (ivars[loo, None] * Aloo)
        ATY = Aloo.T @ (ivars[loo] * ys[loo])
        pars[samp] = np.linalg.solve(ATA, ATY)
    meanpars = np.mean(pars, axis=0)
    diagvars = ((njack - 1) / njack) * np.sum((pars - meanpars[None, :]) ** 2, axis=0)
    return meanpars, diagvars, pars

In [ ]:
nterms = 3
for row in clock_table:
    print(row)
    pars, pars_vars, loos = polynomial_time_delay(row, nterms=nterms)
    print(pars, np.sqrt(pars_vars), loos)
    frequency_tweak = 59. * pars[1]
    print(f"trying frequency tweak {frequency_tweak:.4e}")
    pars, pars_vars, loos = polynomial_time_delay(row, frequency_tweak=frequency_tweak, nterms=nterms) # conceivably a thing?
    print(pars, np.sqrt(pars_vars))